# Torsion Scan Guardian — Vast.ai multi-molecule sweep

Runs the full active-learning pipeline on every molecule in [`data/molecule_library/candidates.csv`](../../data/molecule_library/candidates.csv) (or any subset), on a Vast.ai GPU instance via VSCode Remote-SSH.

**Prerequisite:** [`VastAI_WAY.md`](../../VastAI_WAY.md) steps 1–7 done.

**Sweep cost estimates** on Vast RTX 3090 on-demand (~$0.25/h):
- 3 `todo` molecules: ~30–45 min → **~$0.15**
- 7 `todo` + `candidate` molecules: ~90 min → **~$0.40**
- All 9 molecules: ~2 h → **~$0.50**

Vast.ai is the **cheapest paid GPU option** in this project's catalogue. The catch: marketplace machines vary in reliability — pick `verified=true` + `reliability>0.99` for production.

**REMEMBER**: `vastai destroy instance <ID>` (not stop) when done — from your local terminal.

## 1. Verify the environment

In [ ]:
import os, torch, shutil
print('cwd:', os.getcwd())
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert os.path.exists('scripts/sweep_molecules.py')
assert shutil.which('xtb'), 'xtb not on PATH — see VastAI_WAY.md step 7'
print('xtb:', shutil.which('xtb'))

In [ ]:
# Pre-warm MACE-OFF cache.
from mace.calculators import mace_off
mace_off(model='medium', device='cuda' if torch.cuda.is_available() else 'cpu')
p = os.path.expanduser('~/.cache/mace/MACE-OFF23_medium.model')
assert os.path.exists(p)
print('MACE-OFF cache:', os.path.getsize(p) // 1024, 'KB')

## 2. Inspect the molecule catalog

In [ ]:
import pandas as pd
cat = pd.read_csv('data/molecule_library/candidates.csv')
cat[['name', 'smiles', 'charge', 'heavy_atoms', 'phase', 'why_interesting']]

## 3. Pick the molecules to sweep

In [ ]:
# === EDIT HERE ===
MOLECULES = None                          # e.g. ['glycine_zwitterion', 'biphenyl']; None = use PHASE_FILTER
PHASE_FILTER = ['todo']                   # one or more of 'todo', 'candidate', 'done'
STEPS = 4000
TEMPERATURE = 300
CLOUD_SIZE = 5
MAX_TRIGGERS = 5
BASELINE_ONLY = False
THRESHOLD_OVERRIDE = None
# =================

selected = cat[cat['name'].isin(MOLECULES)] if MOLECULES else cat[cat['phase'].isin(PHASE_FILTER)]
print(f'Will sweep {len(selected)} molecule(s):')
selected[['name', 'smiles', 'heavy_atoms', 'phase']]

## 4. Run the sweep

**STRONGLY RECOMMENDED on Vast:** run this from a `tmux` session in the VSCode terminal instead of from the notebook, so a VSCode/SSH disconnect doesn't kill the sweep. Open a terminal and run:
```bash
tmux new -s sweep
MPLBACKEND=Agg python scripts/sweep_molecules.py \
    --phase-filter todo --steps 4000 --device cuda
# Ctrl-B then d to detach. Reconnect later with `tmux attach -t sweep`.
```
Vast instances on community hosts occasionally drop SSH. tmux is the difference between "I lost 30 min of compute" and "I lost nothing".

In [ ]:
import subprocess, sys

cmd = [sys.executable, 'scripts/sweep_molecules.py',
       '--steps', str(STEPS), '--temperature', str(TEMPERATURE),
       '--cloud-size', str(CLOUD_SIZE), '--max-triggers', str(MAX_TRIGGERS),
       '--device', 'cuda']
if MOLECULES:
    cmd += ['--molecules', *MOLECULES]
else:
    cmd += ['--phase-filter', *PHASE_FILTER]
if BASELINE_ONLY:
    cmd.append('--baseline-only')
if THRESHOLD_OVERRIDE is not None:
    cmd += ['--threshold', str(THRESHOLD_OVERRIDE)]

env = dict(os.environ); env['PYTHONIOENCODING'] = 'utf-8'; env['MPLBACKEND'] = 'Agg'

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding='utf-8', env=env, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\nsweep exit code: {rc}')

## 5. Review summary

In [ ]:
summary = pd.read_csv('runs/sweep/sweep_summary.csv')
cols = ['name', 'status', 'heavy_atoms', 'calibrated_threshold',
        'baseline_wall_s', 'baseline_max_bond_stretch', 'baseline_broken_bonds_final',
        'al_wall_s', 'al_triggers', 'al_labels',
        'al_max_bond_stretch', 'al_broken_bonds_final', 'total_wall_s']
summary[[c for c in cols if c in summary.columns]]

In [ ]:
import matplotlib.pyplot as plt, numpy as np
df = summary[summary['status'] == 'ok'].copy()
names = df['name'].tolist()
x = np.arange(len(names)); w = 0.4
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(x - w/2, df['baseline_max_bond_stretch'], w, label='baseline', color='#7f7f7f')
if 'al_max_bond_stretch' in df.columns:
    axes[0].bar(x + w/2, df['al_max_bond_stretch'], w, label='AL', color='#1f77b4')
axes[0].axhline(1.6, color='#d62728', linestyle='--', lw=0.7, label='bond-break 1.6x')
axes[0].axhline(1.0, color='black', lw=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].set_ylabel('max bond stretch ratio'); axes[0].set_title('Stability (lower = better)')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)
if 'al_triggers' in df.columns:
    axes[1].bar(x, df['al_triggers'].fillna(0), color='#2ca02c')
    axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=30, ha='right')
    axes[1].set_ylabel('AL cycles'); axes[1].set_title('Triggers fired'); axes[1].grid(axis='y', alpha=0.3)
fig.suptitle('Multi-molecule sweep — AL vs baseline (Vast.ai)', y=1.02)
fig.tight_layout()
fig.savefig('figures/sweep_summary_vastai.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Push results + destroy the instance

1. **Push results**: see [`VastAI_WAY.md`](../../VastAI_WAY.md) §10 for the three options (scp + push from local, PAT, SSH key).
2. **DESTROY the instance** from your local terminal: `vastai destroy instance <ID>`. Use `destroy`, not `stop` — destroying frees the disk and stops all billing for that instance.